# Test: Dashboard visualizations (pipeline step 9)

This notebook:

1. **Validates** that dashboard visualization artifacts exist (BupaR, DTW, FP-Growth) for each cohort/age_band.
2. **Tests actual creation** by running the creation scripts for one combination (smoke test), then verifying outputs.

- **BupaR** – Process mining plots (PNG); creation requires R and bupaR.
- **DTW** – Trajectory features CSV and plots (PNG/HTML).
- **FP-Growth** – Itemsets JSON and plots (PNG/HTML).

Run from repo root. Prerequisites for creation: 4_model_data, and (for BupaR) R + bupaR.

In [ ]:
# Setup: repo root and pipeline cohort/age_band combinations
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "py_helpers").exists():
    for p in REPO_ROOT.parents:
        if (p / "py_helpers").exists():
            REPO_ROOT = p
            break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

try:
    from py_helpers.constants import REQUIRED_COHORTS
except ImportError:
    REQUIRED_COHORTS = {
        "opioid_ed": ["13-24", "25-44", "45-54", "55-64"],
        "non_opioid_ed": ["65-74", "75-84", "85-94"],
    }

COMBINATIONS = [(c, ab) for c, bands in REQUIRED_COHORTS.items() for ab in bands]
print(f"Repo root: {REPO_ROOT}")
print(f"Cohort/age_band combinations: {len(COMBINATIONS)}")
for c, ab in COMBINATIONS:
    print(f"  {c} / {ab}")

In [ ]:
def check_bupar_outputs(repo_root: Path, cohort: str, age_band: str) -> dict:
    """Check BupaR outputs: plots under 10c or feature_engineering_outputs."""
    ab_f = age_band.replace("-", "_")
    dirs = [
        repo_root / "10_risk_dashboard" / "visualizations" / "bupar" / "outputs" / cohort / ab_f / "plots",
        repo_root / "5_feature_engineering" / "feature_engineering_outputs" / "5_bupar" / cohort / ab_f / "plots",
    ]
    for d in dirs:
        if d.exists() and list(d.glob("*.png")):
            return {"ok": True, "path": str(d), "count": len(list(d.glob("*.png")))}
    return {"ok": False, "path": str(dirs[0]), "count": 0}


def check_dtw_outputs(repo_root: Path, cohort: str, age_band: str) -> dict:
    """Check DTW outputs: features CSV and plots."""
    ab_f = age_band.replace("-", "_")
    fe_csv = (
        repo_root / "10_risk_dashboard" / "visualizations" / "dtw" / "outputs" / "feature_engineering"
        / f"dtw_added_features_{cohort}_{ab_f}.csv"
    )
    plot_dirs = [
        repo_root / "10_risk_dashboard" / "visualizations" / "dtw" / "outputs" / cohort / ab_f / "plots",
        repo_root / "5_feature_engineering" / "feature_engineering_outputs" / "6_dtw" / cohort / age_band / "plots",
    ]
    csv_ok = fe_csv.exists()
    plot_dir = None
    plot_count = 0
    for d in plot_dirs:
        if d.exists():
            pngs = list(d.glob("*.png"))
            htmls = list(d.glob("*.html"))
            if pngs or htmls:
                plot_dir = d
                plot_count = len(pngs) + len(htmls)
                break
    return {
        "ok": csv_ok and plot_count > 0,
        "csv": csv_ok,
        "plots_path": str(plot_dir) if plot_dir else None,
        "plot_count": plot_count,
    }


def check_fpgrowth_outputs(repo_root: Path, cohort: str, age_band: str) -> dict:
    """Check FP-Growth outputs: itemsets and plots."""
    ab_f = age_band.replace("-", "_")
    itemsets_dir = (
        repo_root / "10_risk_dashboard" / "visualizations" / "fpgrowth" / "outputs" / cohort / "target" / ab_f / "train"
    )
    itemsets_ok = itemsets_dir.exists() and any(itemsets_dir.glob("*_itemsets*.json"))
    plots_dir = (
        repo_root / "5_feature_engineering" / "feature_engineering_outputs" / "4_fpgrowth"
        / cohort / age_band / "plots"
    )
    if not plots_dir.exists():
        plots_dir = repo_root / "10_risk_dashboard" / "visualizations" / "fpgrowth" / "outputs" / cohort / ab_f / "plots"
    plot_count = len(list(plots_dir.glob("*.png"))) + len(list(plots_dir.glob("*.html"))) if plots_dir.exists() else 0
    return {
        "ok": itemsets_ok and plot_count > 0,
        "itemsets": itemsets_ok,
        "plots_path": str(plots_dir) if plots_dir.exists() else None,
        "plot_count": plot_count,
    }


print("Helper functions defined: check_bupar_outputs, check_dtw_outputs, check_fpgrowth_outputs")

In [ ]:
# Run checks for all cohort/age_band combinations
results = []
for cohort, age_band in COMBINATIONS:
    bupar = check_bupar_outputs(REPO_ROOT, cohort, age_band)
    dtw = check_dtw_outputs(REPO_ROOT, cohort, age_band)
    fpg = check_fpgrowth_outputs(REPO_ROOT, cohort, age_band)
    results.append({
        "cohort": cohort,
        "age_band": age_band,
        "bupar": bupar,
        "dtw": dtw,
        "fpgrowth": fpg,
    })

# Summary table
print("Dashboard visuals test results")
print("=" * 80)
print(f"{"Cohort":<16} {"Age":<8} {"BupaR":<8} {"DTW":<8} {"FP-Growth":<10}")
print("-" * 80)
for r in results:
    b = "OK" if r["bupar"]["ok"] else "missing"
    d = "OK" if r["dtw"]["ok"] else "missing"
    f = "OK" if r["fpgrowth"]["ok"] else "missing"
    print(f"{r['cohort']:<16} {r['age_band']:<8} {b:<8} {d:<8} {f:<10}")

bupar_ok = sum(1 for r in results if r["bupar"]["ok"])
dtw_ok = sum(1 for r in results if r["dtw"]["ok"])
fpg_ok = sum(1 for r in results if r["fpgrowth"]["ok"])
total = len(results)
print("-" * 80)
print(f"Total: {bupar_ok}/{total} BupaR, {dtw_ok}/{total} DTW, {fpg_ok}/{total} FP-Growth")

In [ ]:
# Optional: display a sample DTW or BupaR plot if available
from IPython.display import Image, display

sample_shown = False
for r in results:
    if sample_shown:
        break
    # Prefer DTW trajectory plot
    d = r["dtw"]
    if d.get("plots_path"):
        plot_path = Path(d["plots_path"])
        pngs = list(plot_path.glob("*.png"))
        if pngs:
            print(f"Sample DTW plot: {r['cohort']} / {r['age_band']}")
            display(Image(filename=str(pngs[0]), width=600))
            sample_shown = True
            break
    # Else BupaR
    b = r["bupar"]
    if b.get("path"):
        p = Path(b["path"])
        if p.exists():
            pngs = list(p.glob("*.png"))
            if pngs:
                print(f"Sample BupaR plot: {r['cohort']} / {r['age_band']}")
                display(Image(filename=str(pngs[0]), width=600))
                sample_shown = True

if not sample_shown:
    print("No local plot found to display. Run 4_dashboard_visuals.ipynb first.")

## Test actual creation (BupaR, DTW, FP-Growth)

The cells below **run the creation scripts** for one cohort/age_band (smoke test) and then verify that outputs were produced. Prerequisites: **4_model_data** and (for BupaR) **R + bupaR**; run from repo root.

- **BupaR:** `create_bupar_visuals.py --cohort-name ... --age-band ...`
- **DTW:** `create_dtw_features.py` then `create_dtw_visuals.py`
- **FP-Growth:** `create_fpgrowth_visuals.py --cohort-name ... --age-band ...`

In [ ]:
# Config: which cohort/age_band to use for creation test (one combination to keep runtime reasonable)
TEST_CREATION_COHORT, TEST_CREATION_AGE_BAND = COMBINATIONS[0]  # e.g. opioid_ed / 13-24
# Set to True to force re-run even if outputs exist
TEST_CREATION_FORCE = False

import subprocess
import os

VISUAL_ROOT = REPO_ROOT / "10_risk_dashboard" / "visualizations"
BUPAR_SCRIPT = VISUAL_ROOT / "bupar" / "create_bupar_visuals.py"
DTW_FEATURES_SCRIPT = VISUAL_ROOT / "dtw" / "create_dtw_features.py"
DTW_VISUALS_SCRIPT = VISUAL_ROOT / "dtw" / "create_dtw_visuals.py"
FPGROWTH_SCRIPT = VISUAL_ROOT / "fpgrowth" / "create_fpgrowth_visuals.py"

# Symlinks 10b/10c/10d at repo root point to 10_risk_dashboard/visualizations; create if missing
for name in ["10c_bupaR_dashboard_visual", "10b_fpgrowth_dashboard_visual", "10d_dtw_dashboard_visual"]:
    if not (REPO_ROOT / name).exists():
        print(f"Note: {name} not at repo root (optional; create symlink for outputs there)")

print(f"Creation test combination: {TEST_CREATION_COHORT} / {TEST_CREATION_AGE_BAND}")
print(f"Force re-run: {TEST_CREATION_FORCE}")
print()

In [ ]:
# Run BupaR, DTW, and FP-Growth creation for the test combination
force_flag = ["--force"] if TEST_CREATION_FORCE else []
c, ab = TEST_CREATION_COHORT, TEST_CREATION_AGE_BAND
creation_results = {}

# 1. BupaR (requires R and bupaR)
print("1. BupaR: create_bupar_visuals.py ...")
if BUPAR_SCRIPT.exists():
    r = subprocess.run(
        [sys.executable, str(BUPAR_SCRIPT), "--cohort-name", c, "--age-band", ab] + force_flag,
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
        timeout=600,
    )
    creation_results["bupar"] = {"returncode": r.returncode, "ok": r.returncode == 0}
    if r.returncode != 0:
        print(f"   stderr: {r.stderr[:500] if r.stderr else 'none'}")
    else:
        print("   OK")
else:
    creation_results["bupar"] = {"returncode": -1, "ok": False}
    print("   SKIP (script not found)")
print()

# 2. DTW (features then visuals)
print("2. DTW: create_dtw_features.py then create_dtw_visuals.py ...")
if DTW_FEATURES_SCRIPT.exists() and DTW_VISUALS_SCRIPT.exists():
    r1 = subprocess.run(
        [sys.executable, str(DTW_FEATURES_SCRIPT), "--cohort", c, "--age_band", ab] + force_flag,
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
        timeout=600,
    )
    if r1.returncode != 0:
        creation_results["dtw"] = {"returncode": r1.returncode, "ok": False}
        print(f"   create_dtw_features failed: {r1.stderr[:300] if r1.stderr else 'none'}")
    else:
        r2 = subprocess.run(
            [sys.executable, str(DTW_VISUALS_SCRIPT), "--cohort-name", c, "--age-band", ab, "--project-root", str(REPO_ROOT)] + force_flag,
            cwd=str(REPO_ROOT),
            capture_output=True,
            text=True,
            timeout=600,
        )
        creation_results["dtw"] = {"returncode": r2.returncode, "ok": r2.returncode == 0}
        if r2.returncode != 0:
            print(f"   create_dtw_visuals stderr: {r2.stderr[:500] if r2.stderr else 'none'}")
        else:
            print("   OK")
else:
    creation_results["dtw"] = {"returncode": -1, "ok": False}
    print("   SKIP (script not found)")
print()

# 3. FP-Growth
print("3. FP-Growth: create_fpgrowth_visuals.py ...")
if FPGROWTH_SCRIPT.exists():
    r = subprocess.run(
        [sys.executable, str(FPGROWTH_SCRIPT), "--cohort-name", c, "--age-band", ab] + force_flag,
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
        timeout=900,
    )
    creation_results["fpgrowth"] = {"returncode": r.returncode, "ok": r.returncode == 0}
    if r.returncode != 0:
        print(f"   stderr: {r.stderr[:500] if r.stderr else 'none'}")
    else:
        print("   OK")
else:
    creation_results["fpgrowth"] = {"returncode": -1, "ok": False}
    print("   SKIP (script not found)")

print()
print("Creation test summary:", {k: ("PASS" if v["ok"] else "FAIL") for k, v in creation_results.items()})

In [ ]:
# Verify: after creation, outputs exist for the test combination
c, ab = TEST_CREATION_COHORT, TEST_CREATION_AGE_BAND
bupar = check_bupar_outputs(REPO_ROOT, c, ab)
dtw = check_dtw_outputs(REPO_ROOT, c, ab)
fpg = check_fpgrowth_outputs(REPO_ROOT, c, ab)

print("Output verification after creation:")
print(f"  BupaR:    {'OK' if bupar['ok'] else 'MISSING'}  (plots: {bupar.get('count', 0)})")
print(f"  DTW:      {'OK' if dtw['ok'] else 'MISSING'}  (csv={dtw.get('csv', False)}, plots={dtw.get('plot_count', 0)})")
print(f"  FP-Growth:{'OK' if fpg['ok'] else 'MISSING'}  (itemsets={fpg.get('itemsets', False)}, plots={fpg.get('plot_count', 0)})")

# Show one created plot if available
from IPython.display import Image, display
for name, res, path_key in [
    ("DTW", dtw, "plots_path"),
    ("BupaR", bupar, "path"),
    ("FP-Growth", fpg, "plots_path"),
]:
    p = res.get(path_key)
    if p and Path(p).exists():
        pngs = list(Path(p).glob("*.png"))
        if pngs:
            print(f"\nSample {name} plot (created):")
            display(Image(filename=str(pngs[0]), width=560))
            break

## Interpretation

- **OK** = at least one expected output (plots or features) found for that visualization type and cohort/age_band.
- **missing** = no local outputs found; run [4_dashboard_visuals.ipynb](../4_dashboard_visuals.ipynb) for that combination or check paths in `10_risk_dashboard/visualizations/` (BupaR, DTW, FP-Growth).
- Symlinks `10b_fpgrowth_dashboard_visual`, `10c_bupaR_dashboard_visual`, `10d_dtw_dashboard_visual` at repo root point to `10_risk_dashboard/visualizations/`; if missing on Windows, create junctions or use the `5_feature_engineering/feature_engineering_outputs/` paths.